In [ ]:
# 04 - Section 5 Statistical Analysis

This notebook reproduces the core Section 5 statistical-analysis pipeline used in the manuscript.

It reads the condition-family curve outputs generated by:

`03_condition_family_visualization.ipynb`

Expected input files:

`outputs/03_condition_family_visualization/family_*/family_curves.csv`

The notebook performs:

- SoC-domain masking and robustification
- Hampel outlier filtering
- optional running-median smoothing
- canonical voltage-shift contrasts
- practical-significance bucket assignment
- point-level factorial screening ANOVA with SoC as a covariate
- output generation for downstream calibration and evaluation notebooks

Important scope note: point-level ANOVA is used only as a dense screening and variance-decomposition summary. The dependence-reduced band-level HC3 analysis and aggressive 2C simple-effect summaries should be added as separate cells if not already generated by another notebook.

In [ ]:
# ============================================================
# Setup and configuration
# ============================================================

import os
import glob
import json
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

# Input generated by 03_condition_family_visualization.ipynb
IN_GLOB = "outputs/03_condition_family_visualization/family_*/family_curves.csv"

# Output directory for Section 5
OUT_DIR = "outputs/04_section5_statistical_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

SOC_BINS = [0, 20, 60, 90, 100]
SOC_LABELS = ["0-20", "20-60", "60-90", "90-100"]

# Canonical inferential bands used in the manuscript
USE_SOC_BANDS = ["0-20", "20-60", "60-90"]

TEMP_LEVELS = [10, 25, 45, 60]

# Column convention inherited from the processed files:
#   Rate        -> charge cut-off current level: C/5 or C/40
#   Charge_Rate -> discharge-rate/rate-history level: 0.7C, 1C, or 2C
RATE_LEVELS = ["C/40", "C/5"]
CHARGE_LEVELS = ["0.7C", "1C", "2C"]

TOP_MASK_FRAC = 0.05

HAMPEL_K_SIGMA = 3.0
HAMPEL_WIN = 31

DO_SMOOTH_MEDIAN = True
SMOOTH_WIN = 7

MONOTONIZE_SOC = False

THRESH_SMALL = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE = 0.15

RNG = np.random.default_rng(42)

print("Input glob:")
print(IN_GLOB)
print("\nOutput directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# Condition map: Condition_ID -> physical factor tuple
# ============================================================
# Tuple format:
#   Condition_ID: (Temperature, charge cut-off current, discharge-rate/rate-history)

COND_MAP = {
    1:  (10, "C/5",  "0.7C"),  2:  (10, "C/40", "0.7C"),
    3:  (10, "C/5",  "1C"),    4:  (10, "C/40", "1C"),
    5:  (10, "C/5",  "2C"),    6:  (10, "C/40", "2C"),
    7:  (25, "C/5",  "0.7C"),  8:  (25, "C/40", "0.7C"),
    9:  (25, "C/5",  "1C"),    10: (25, "C/40", "1C"),
    11: (25, "C/5",  "2C"),    12: (25, "C/40", "2C"),
    13: (45, "C/5",  "0.7C"),  14: (45, "C/40", "0.7C"),
    15: (45, "C/5",  "1C"),    16: (45, "C/40", "1C"),
    17: (45, "C/5",  "2C"),    18: (45, "C/40", "2C"),
    19: (60, "C/5",  "0.7C"),  20: (60, "C/40", "0.7C"),
    21: (60, "C/5",  "1C"),    22: (60, "C/40", "1C"),
    23: (60, "C/5",  "2C"),    24: (60, "C/40", "2C"),
}

condition_map_df = pd.DataFrame(
    [
        {
            "Condition_ID": cid,
            "Temperature": vals[0],
            "Rate": vals[1],
            "Charge_Rate": vals[2],
        }
        for cid, vals in COND_MAP.items()
    ]
)

display(condition_map_df)

In [ ]:
# ============================================================
# Safe numeric helpers
# ============================================================

def nanmedian_safe(a):
    a = np.asarray(a)
    if a.size == 0 or not np.isfinite(a).any():
        return np.nan
    return np.nanmedian(a)


def mad_safe(a, med=None):
    a = np.asarray(a)
    if a.size == 0 or not np.isfinite(a).any():
        return np.nan

    if med is None:
        med = nanmedian_safe(a)

    if not np.isfinite(med):
        return np.nan

    dev = np.abs(a - med)
    mad = nanmedian_safe(dev)

    return mad

In [ ]:
# ============================================================
# Input normalization helpers
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    def pick(candidates=None, contains=None):
        if candidates:
            for cand in candidates:
                key = cand.lower()
                if key in lower:
                    return lower[key]

        if contains:
            for c in cols:
                if contains.lower() in c.lower():
                    return c

        return None

    colmap = {}

    soc_col = pick(
        candidates=["SoC", "soc", "state_of_charge", "soc_%"],
        contains="soc"
    )
    if soc_col:
        colmap[soc_col] = "SoC"

    temp_col = pick(
        candidates=["Temperature", "Temp", "temperature_c", "ambient_temperature", "T"],
        contains="temp"
    )
    if temp_col:
        colmap[temp_col] = "Temperature"

    rate_col = pick(
        candidates=["Rate", "Discharge_Rate", "Test_Rate", "Discharge"],
        contains="rate"
    )
    if rate_col:
        colmap[rate_col] = "Rate"

    chg_col = pick(
        candidates=["Charge_Rate", "ChargeRate", "Charge-Rate", "CR", "Charge"],
        contains="charge"
    )
    if chg_col:
        colmap[chg_col] = "Charge_Rate"

    cond_col = pick(
        candidates=["Condition_ID", "ConditionId", "CondID", "Condition"],
        contains="condition"
    )
    if cond_col:
        colmap[cond_col] = "Condition_ID"

    if "V_median" in df.columns:
        colmap["V_median"] = "Voltage"
    else:
        if ("V_q25" in df.columns) and ("V_q75" in df.columns):
            df["Voltage"] = (
                pd.to_numeric(df["V_q25"], errors="coerce")
                + pd.to_numeric(df["V_q75"], errors="coerce")
            ) / 2.0
        else:
            volt_col = pick(
                candidates=["Voltage", "VoltageV", "V", "Cell_Voltage", "Voltage (V)", "volt"],
                contains="volt"
            )
            if volt_col:
                colmap[volt_col] = "Voltage"

    df2 = df.rename(columns=colmap)

    if "SoC" in df2.columns:
        df2["SoC"] = pd.to_numeric(df2["SoC"], errors="coerce")

    if "Voltage" in df2.columns:
        df2["Voltage"] = pd.to_numeric(df2["Voltage"], errors="coerce")

    return df2


def coerce_and_impute_from_condition(df: pd.DataFrame) -> pd.DataFrame:
    if "Condition_ID" in df.columns:
        df["Condition_ID"] = pd.to_numeric(
            df["Condition_ID"],
            errors="coerce"
        ).astype("Int64")

    required_factor_cols = ["Temperature", "Rate", "Charge_Rate"]

    for c in required_factor_cols:
        if c not in df.columns:
            df[c] = np.nan

    if "Condition_ID" in df.columns:
        mask_missing = (
            df["Temperature"].isna()
            | df["Rate"].isna()
            | df["Charge_Rate"].isna()
        )

        if mask_missing.any():
            def fill_row(row):
                cid = row.get("Condition_ID", pd.NA)

                try:
                    cid = int(cid)
                except Exception:
                    return row

                if cid in COND_MAP:
                    T, R, CR = COND_MAP[cid]

                    if pd.isna(row["Temperature"]):
                        row["Temperature"] = T

                    if pd.isna(row["Rate"]):
                        row["Rate"] = R

                    if pd.isna(row["Charge_Rate"]):
                        row["Charge_Rate"] = CR

                return row

            df = df.apply(fill_row, axis=1)

    if "Temperature" in df.columns:
        df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce")

    return df

In [ ]:
# ============================================================
# Load and concatenate family-curve CSV files
# ============================================================

def load_concat(in_glob=IN_GLOB) -> pd.DataFrame:
    files = sorted(glob.glob(in_glob))

    if not files:
        raise FileNotFoundError(f"No files found for input glob: {in_glob}")

    dfs = []

    for fp in files:
        if fp.lower().endswith(".csv"):
            df = pd.read_csv(fp)
        else:
            df = pd.read_excel(fp)

        df["__source_file__"] = os.path.basename(fp)
        df["__source_path__"] = fp

        dfs.append(df)

    raw = pd.concat(dfs, ignore_index=True)

    df = normalize_columns(raw.copy())
    df = coerce_and_impute_from_condition(df)

    if "quantile" in df.columns:
        med_mask = df["quantile"].astype(str).str.lower().eq("median")
        if med_mask.any():
            df = df[med_mask].copy()

    required = ["SoC", "Voltage", "Temperature", "Rate", "Charge_Rate"]

    for c in required:
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].replace(
                {"nan": np.nan, "NaN": np.nan, "None": np.nan}
            )

    df = df.dropna(subset=required)

    if df.empty:
        diag_path = os.path.join(OUT_DIR, "load_diag.txt")
        with open(diag_path, "w", encoding="utf-8") as f:
            f.write("Data became empty after dropna.\n")
            f.write("Raw columns:\n")
            f.write(str(list(raw.columns)) + "\n")

        return df

    df["Temperature"] = pd.to_numeric(
        df["Temperature"],
        errors="coerce"
    ).astype(int)

    df["Rate"] = pd.Categorical(
        df["Rate"],
        categories=RATE_LEVELS,
        ordered=True
    )

    df["Charge_Rate"] = pd.Categorical(
        df["Charge_Rate"],
        categories=CHARGE_LEVELS,
        ordered=True
    )

    df["Temperature"] = pd.Categorical(
        df["Temperature"],
        categories=TEMP_LEVELS,
        ordered=True
    )

    df["SoC_band"] = pd.cut(
        df["SoC"].clip(0, 100),
        bins=SOC_BINS,
        labels=SOC_LABELS,
        include_lowest=True
    )

    return df


df_raw = load_concat()

print("Loaded family-curve data:")
print(f"Rows: {len(df_raw):,}")
print(f"Covered temperatures: {df_raw['Temperature'].dropna().unique()}")
print(f"Covered charge cut-off levels: {df_raw['Rate'].dropna().unique()}")
print(f"Covered discharge-rate/rate-history levels: {df_raw['Charge_Rate'].dropna().unique()}")

display(df_raw.head())

In [ ]:
# ============================================================
# Masking and robustification
# ============================================================

def mask_top_end(df: pd.DataFrame, top_frac=TOP_MASK_FRAC) -> pd.Series:
    threshold = 100.0 * (1.0 - float(top_frac))
    return df["SoC"] >= threshold


def hampel_mask_1d(y: np.ndarray, k=HAMPEL_K_SIGMA, win=HAMPEL_WIN) -> np.ndarray:
    if win % 2 == 0:
        win += 1

    y = np.asarray(y, dtype=float)
    n = len(y)

    if n == 0:
        return np.zeros(0, dtype=bool)

    half = win // 2
    mask = np.zeros(n, dtype=bool)

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        window = y[lo:hi]
        med = nanmedian_safe(window)
        mad = mad_safe(window, med)

        if not np.isfinite(med) or not np.isfinite(mad):
            mask[i] = False
            continue

        scale = 1.4826 * mad if mad > 0 else np.nanstd(window)

        if not np.isfinite(scale) or scale == 0:
            mask[i] = False
            continue

        mask[i] = abs(y[i] - med) > k * scale

    return mask


def apply_masks_and_smooth(df: pd.DataFrame):
    d = df.copy().reset_index(drop=True)

    d["mask_top"] = mask_top_end(d)
    d["mask_hampel"] = False

    group_cols = ["Temperature", "Rate", "Charge_Rate"]
    reports = []

    def smooth_median(arr, win=SMOOTH_WIN):
        if win % 2 == 0:
            win += 1

        arr = np.asarray(arr, dtype=float)
        n = len(arr)

        if n == 0 or win <= 1:
            return arr

        half = win // 2
        out = arr.copy()

        for i in range(n):
            lo = max(0, i - half)
            hi = min(n, i + half + 1)

            window = arr[lo:hi]
            med = nanmedian_safe(window)

            if np.isfinite(med):
                out[i] = med

        return out

    d = d.sort_values(group_cols + ["SoC"]).reset_index(drop=True)

    for key, g in d.groupby(group_cols, sort=False, dropna=False):
        idx = g.index.to_numpy()

        hmask = hampel_mask_1d(
            g["Voltage"].to_numpy(),
            k=HAMPEL_K_SIGMA,
            win=HAMPEL_WIN
        )

        d.loc[idx, "mask_hampel"] = hmask

        v = g["Voltage"].to_numpy().astype(float)
        v_clean = v.copy()

        v_clean[hmask | d.loc[idx, "mask_top"].to_numpy()] = np.nan

        if DO_SMOOTH_MEDIAN:
            keep = ~np.isnan(v_clean)
            if keep.any():
                v_clean[keep] = smooth_median(v_clean[keep], win=SMOOTH_WIN)

        d.loc[idx, "Voltage_clean"] = v_clean

        reports.append({
            "Temperature": key[0],
            "Rate": key[1],
            "Charge_Rate": key[2],
            "n_total": int(len(g)),
            "n_mask_top": int(d.loc[idx, "mask_top"].sum()),
            "n_mask_hampel": int(np.sum(hmask)),
        })

    mask_report = pd.DataFrame.from_records(reports)

    if mask_report.empty:
        mask_report = pd.DataFrame(
            columns=[
                "Temperature", "Rate", "Charge_Rate",
                "n_total", "n_mask_top", "n_mask_hampel", "mask_ratio_%"
            ]
        )
    else:
        denom = mask_report["n_total"].replace(0, np.nan)
        ratio = (
            100.0
            * (
                mask_report["n_mask_top"].fillna(0)
                + mask_report["n_mask_hampel"].fillna(0)
            )
            / denom
        )
        mask_report["mask_ratio_%"] = ratio.fillna(0.0)

    d["SoC_band"] = pd.cut(
        d["SoC"].clip(0, 100),
        bins=SOC_BINS,
        labels=SOC_LABELS,
        include_lowest=True
    )

    return d, mask_report


clean, mask_report = apply_masks_and_smooth(df_raw)

clean_path = os.path.join(OUT_DIR, "tidy_clean.csv")
mask_report_path = os.path.join(OUT_DIR, "mask_report_by_group.csv")

clean.to_csv(clean_path, index=False)
mask_report.to_csv(mask_report_path, index=False)

print("Masking and robustification completed.")
print(f"Clean rows: {len(clean):,}")
print(f"Saved: {clean_path}")
print(f"Saved: {mask_report_path}")

display(mask_report.head())

In [ ]:
# ============================================================
# Effect-size and practical-bucket helpers
# ============================================================

@dataclass
class DiffSummary:
    contrast: str
    baseline: str
    levelset: dict
    soc_band: str
    n_baseline: int
    n_contrast: int
    mean_diff: float
    ci_low: float
    ci_high: float
    cliffs_delta: float
    glass_delta: float
    practical_bucket: str


def practical_bucket(delta_abs: float) -> str:
    if np.isnan(delta_abs):
        return "NA"
    if delta_abs >= THRESH_LARGE:
        return "large"
    if delta_abs >= THRESH_MEDIUM:
        return "moderate"
    if delta_abs >= THRESH_SMALL:
        return "small"
    return "negligible"


def bootstrap_ci(diff_samples: np.ndarray, alpha=0.05):
    if diff_samples.size == 0:
        return np.nan, np.nan

    lo, hi = np.quantile(diff_samples, [alpha / 2, 1 - alpha / 2])

    return float(lo), float(hi)


def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    nx, ny = len(x), len(y)

    if nx == 0 or ny == 0:
        return np.nan

    diffs = np.subtract.outer(y, x)

    return float((np.sum(diffs > 0) - np.sum(diffs < 0)) / (nx * ny))


def glass_delta(x_baseline, y_contrast):
    xb = np.asarray(x_baseline)
    yc = np.asarray(y_contrast)

    if xb.size < 2 or yc.size < 1:
        return np.nan

    s = np.std(xb, ddof=1)

    if s == 0:
        return np.nan

    return float((np.mean(yc) - np.mean(xb)) / s)

In [ ]:
# ============================================================
# Delta-V contrast table
# ============================================================

def delta_table(
    df: pd.DataFrame,
    factor: str,
    baseline,
    contrast,
    fixed: dict,
    use_bands=USE_SOC_BANDS,
    n_boot=1000,
    use_clean_voltage=True
) -> pd.DataFrame:

    needed = {"Rate", "Charge_Rate", "Temperature", "SoC_band"}

    if not needed.issubset(df.columns):
        return pd.DataFrame()

    vcol = "Voltage_clean" if use_clean_voltage and "Voltage_clean" in df.columns else "Voltage"

    sub = df.copy()

    for key, value in fixed.items():
        sub = sub[sub[key] == value]

    if (
        sub.empty
        or sub[sub[factor] == baseline].empty
        or sub[sub[factor] == contrast].empty
    ):
        return pd.DataFrame.from_records([
            {
                "contrast": str(contrast),
                "baseline": str(baseline),
                "levelset": fixed,
                "soc_band": band,
                "n_baseline": 0,
                "n_contrast": 0,
                "mean_diff": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
                "cliffs_delta": np.nan,
                "glass_delta": np.nan,
                "practical_bucket": "NA",
            }
            for band in use_bands
        ])

    records = []

    for band in use_bands:
        sb = sub[sub["SoC_band"] == band]

        xb = sb[sb[factor] == baseline][vcol].dropna().to_numpy()
        yc = sb[sb[factor] == contrast][vcol].dropna().to_numpy()

        if xb.size == 0 or yc.size == 0:
            md = np.nan
            ci = (np.nan, np.nan)
            cd = np.nan
            gd = np.nan
            bucket = "NA"

        else:
            md = float(np.mean(yc) - np.mean(xb))

            diffs = []

            for _ in range(n_boot):
                b = RNG.choice(xb, size=xb.size, replace=True)
                c = RNG.choice(yc, size=yc.size, replace=True)
                diffs.append(np.mean(c) - np.mean(b))

            ci = bootstrap_ci(np.asarray(diffs))
            cd = cliffs_delta(xb, yc)
            gd = glass_delta(xb, yc)
            bucket = practical_bucket(abs(md))

        rec = DiffSummary(
            contrast=str(contrast),
            baseline=str(baseline),
            levelset=fixed,
            soc_band=band,
            n_baseline=int(xb.size),
            n_contrast=int(yc.size),
            mean_diff=md,
            ci_low=ci[0],
            ci_high=ci[1],
            cliffs_delta=cd,
            glass_delta=gd,
            practical_bucket=bucket,
        ).__dict__

        records.append(rec)

    return pd.DataFrame.from_records(records)

In [ ]:
# ============================================================
# Canonical Delta-V contrasts
# ============================================================

# Charge cut-off current contrast:
# C/40 - C/5 at T = 25 °C and discharge-rate/rate-history level = 1C
delta_cutoff_T25_DR1C = delta_table(
    clean,
    factor="Rate",
    baseline="C/5",
    contrast="C/40",
    fixed={"Temperature": 25, "Charge_Rate": "1C"}
)

# Mild discharge-rate/rate-history contrast:
# 0.7C - 1C at T = 25 °C and charge cut-off current = C/40
delta_ratehist_T25_R_C40 = delta_table(
    clean,
    factor="Charge_Rate",
    baseline="1C",
    contrast="0.7C",
    fixed={"Temperature": 25, "Rate": "C/40"}
)

# Thermal contrast:
# 60 °C - 25 °C at charge cut-off current = C/40
# and discharge-rate/rate-history level = 1C
delta_temp_R_C40_DR1C = delta_table(
    clean,
    factor="Temperature",
    baseline=25,
    contrast=60,
    fixed={"Rate": "C/40", "Charge_Rate": "1C"}
)

paths = {
    "delta_cutoff_T25_DR1C.csv": delta_cutoff_T25_DR1C,
    "delta_ratehist_T25_R_C40.csv": delta_ratehist_T25_R_C40,
    "delta_temp_R_C40_DR1C.csv": delta_temp_R_C40_DR1C,
}

for filename, df_out in paths.items():
    out_path = os.path.join(OUT_DIR, filename)
    df_out.to_csv(out_path, index=False)
    print("Saved:", out_path)

print("\nCharge cut-off current contrast:")
display(delta_cutoff_T25_DR1C)

print("\nMild discharge-rate/rate-history contrast:")
display(delta_ratehist_T25_R_C40)

print("\nThermal contrast:")
display(delta_temp_R_C40_DR1C)

In [ ]:
# ============================================================
# Canonical contrast grids for heat-map and bucket summaries
# ============================================================

rows = []

for T in TEMP_LEVELS:
    dT = delta_table(
        clean,
        factor="Rate",
        baseline="C/5",
        contrast="C/40",
        fixed={"Temperature": T, "Charge_Rate": "1C"}
    )
    dT["Temperature"] = T
    rows.append(dT)

cutoff_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
cutoff_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
cutoff_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_cutoff_grid.csv"),
    index=False
)

rows = []

for T in TEMP_LEVELS:
    dT = delta_table(
        clean,
        factor="Charge_Rate",
        baseline="1C",
        contrast="0.7C",
        fixed={"Temperature": T, "Rate": "C/40"}
    )
    dT["Temperature"] = T
    rows.append(dT)

ratehist_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
ratehist_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
ratehist_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_ratehist_grid.csv"),
    index=False
)

rows = []

for dr in CHARGE_LEVELS:
    ddr = delta_table(
        clean,
        factor="Temperature",
        baseline=25,
        contrast=60,
        fixed={"Rate": "C/40", "Charge_Rate": dr}
    )
    ddr["Charge_Rate"] = dr
    rows.append(ddr)

temp_grid = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
temp_grid.rename(columns={"soc_band": "SoC_band"}, inplace=True)
temp_grid.to_csv(
    os.path.join(OUT_DIR, "heatmap_delta_temp_grid.csv"),
    index=False
)

print("Saved canonical contrast grids:")
print(" - heatmap_delta_cutoff_grid.csv")
print(" - heatmap_delta_ratehist_grid.csv")
print(" - heatmap_delta_temp_grid.csv")

In [ ]:
# ============================================================
# Three-way point-level ANOVA with SoC as covariate
# ============================================================

def three_way_anova(
    df: pd.DataFrame,
    use_bands=USE_SOC_BANDS,
    use_clean_voltage=True
) -> pd.DataFrame:

    if df is None or df.empty:
        out = pd.DataFrame()
        out.to_csv(os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv"))
        with open(os.path.join(OUT_DIR, "anova_model_summary.txt"), "w", encoding="utf-8") as f:
            f.write("Empty dataframe; ANOVA skipped.\n")
        return out

    vcol = "Voltage_clean" if use_clean_voltage and "Voltage_clean" in df.columns else "Voltage"

    d = df[df["SoC_band"].isin(use_bands)].copy()

    if d.empty or d[vcol].dropna().size < 8:
        out = pd.DataFrame()
        out.to_csv(os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv"))
        with open(os.path.join(OUT_DIR, "anova_model_summary.txt"), "w", encoding="utf-8") as f:
            f.write("Insufficient rows for ANOVA after filtering.\n")
        return out

    d["SoC_cont"] = d["SoC"] / 100.0

    for c in ["Temperature", "Rate", "Charge_Rate"]:
        d[c] = d[c].astype("category")

    try:
        model = smf.ols(
            f"{vcol} ~ C(Temperature)*C(Rate)*C(Charge_Rate) + SoC_cont",
            data=d
        ).fit()

        aov = anova_lm(model, typ=2)

        aov_path = os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv")
        summary_path = os.path.join(OUT_DIR, "anova_model_summary.txt")

        aov.to_csv(aov_path)

        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(model.summary().as_text())

        print("ANOVA completed.")
        print("Saved:", aov_path)
        print("Saved:", summary_path)

        return aov

    except Exception as e:
        summary_path = os.path.join(OUT_DIR, "anova_model_summary.txt")
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(f"ANOVA failed: {e}\n")

        pd.DataFrame().to_csv(
            os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC.csv")
        )

        print("ANOVA failed:", e)

        return pd.DataFrame()


anova_table = three_way_anova(clean)

display(anova_table)

In [ ]:
# ============================================================
# Run summary
# ============================================================

summary = {
    "n_rows_clean": int(len(clean)),
    "soc_bands": USE_SOC_BANDS,
    "temp_levels": TEMP_LEVELS,
    "charge_cutoff_levels_Rate_column": RATE_LEVELS,
    "discharge_rate_history_levels_Charge_Rate_column": CHARGE_LEVELS,
    "top_mask_frac": TOP_MASK_FRAC,
    "hampel_k_sigma": HAMPEL_K_SIGMA,
    "hampel_win": HAMPEL_WIN,
    "smooth_median": DO_SMOOTH_MEDIAN,
    "smooth_win": SMOOTH_WIN,
    "practical_thresholds_V": {
        "small": THRESH_SMALL,
        "moderate": THRESH_MEDIUM,
        "large": THRESH_LARGE,
    },
}

summary_path = os.path.join(OUT_DIR, "run_summary.json")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Section 5 core statistical-analysis pipeline completed.")
print("Outputs directory:", OUT_DIR)
print("Saved:", summary_path)

In [ ]:
## HC3, band-level robustness, and practical-bucket validation

This extension adds the robustness and bucket-summary outputs used in the manuscript.

It assumes that the previous Section 5 cells have already generated:

- `tidy_clean.csv`
- `delta_cutoff_T25_DR1C.csv`
- `delta_ratehist_T25_R_C40.csv`
- `delta_temp_R_C40_DR1C.csv`

The extension provides:

- point-level HC3-robust factorial screening
- de-duplicated band-level cluster-robust sensitivity analysis by `Condition_ID`
- practical-bucket validation from numerical `mean_diff` values

Important interpretation note: HC3 protects against heteroskedasticity but does not remove repeated-measure, serial, or within-curve dependence. Therefore, the point-level HC3 output should be interpreted as factorial screening, while the band-level cluster-robust output is used as a dependence-reduced sensitivity check.

In [ ]:
# ============================================================
# HC3, band-level robustness, and bucket-summary setup
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# Use the same Section 5 output directory defined earlier.
OUT_DIR = "outputs/04_section5_statistical_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

TIDY_PATH = os.path.join(OUT_DIR, "tidy_clean.csv")

# Canonical delta files generated earlier in this notebook.
DELTA_CUTOFF = os.path.join(OUT_DIR, "delta_cutoff_T25_DR1C.csv")
DELTA_RATEHIST = os.path.join(OUT_DIR, "delta_ratehist_T25_R_C40.csv")
DELTA_TEMP = os.path.join(OUT_DIR, "delta_temp_R_C40_DR1C.csv")

USE_SOC_BANDS = ["0-20", "20-60", "60-90"]

TEMP_LEVELS = [10, 25, 45, 60]

# Internal column convention:
#   Rate        -> charge cut-off current level
#   Charge_Rate -> discharge-rate/rate-history level
RATE_LEVELS = ["C/40", "C/5"]
CR_LEVELS = ["0.7C", "1C", "2C"]

THRESH_SMALL = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE = 0.15

COND_MAP = {
    1:  (10, "C/5",  "0.7C"),  2:  (10, "C/40", "0.7C"),
    3:  (10, "C/5",  "1C"),    4:  (10, "C/40", "1C"),
    5:  (10, "C/5",  "2C"),    6:  (10, "C/40", "2C"),
    7:  (25, "C/5",  "0.7C"),  8:  (25, "C/40", "0.7C"),
    9:  (25, "C/5",  "1C"),    10: (25, "C/40", "1C"),
    11: (25, "C/5",  "2C"),    12: (25, "C/40", "2C"),
    13: (45, "C/5",  "0.7C"),  14: (45, "C/40", "0.7C"),
    15: (45, "C/5",  "1C"),    16: (45, "C/40", "1C"),
    17: (45, "C/5",  "2C"),    18: (45, "C/40", "2C"),
    19: (60, "C/5",  "0.7C"),  20: (60, "C/40", "0.7C"),
    21: (60, "C/5",  "1C"),    22: (60, "C/40", "1C"),
    23: (60, "C/5",  "2C"),    24: (60, "C/40", "2C"),
}

INV_COND_MAP = {(T, R, CR): cid for cid, (T, R, CR) in COND_MAP.items()}

print("HC3/bucket extension inputs:")
for p in [TIDY_PATH, DELTA_CUTOFF, DELTA_RATEHIST, DELTA_TEMP]:
    print(f" - {p}: {os.path.exists(p)}")

In [ ]:
# ============================================================
# Robustness helper functions
# ============================================================

def ensure_exists(path: str) -> None:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")


def find_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def standard_band_value(x) -> str:
    s = str(x).strip()
    s = s.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    s = s.replace(", ", "-").replace(",", "-")
    s = s.replace("0.0-20.0", "0-20")
    s = s.replace("20.0-60.0", "20-60")
    s = s.replace("60.0-90.0", "60-90")
    s = s.replace("90.0-100.0", "90-100")
    return s


def practical_bucket_checked(delta_abs: float) -> str:
    if pd.isna(delta_abs):
        return "NA"

    delta_abs = float(abs(delta_abs))

    if delta_abs >= THRESH_LARGE:
        return "large"
    if delta_abs >= THRESH_MEDIUM:
        return "moderate"
    if delta_abs >= THRESH_SMALL:
        return "small"

    return "negligible"


def pick_voltage_col(df: pd.DataFrame) -> str:
    if "Voltage_clean" in df.columns:
        return "Voltage_clean"

    if "Voltage" in df.columns:
        return "Voltage"

    if "V_median" in df.columns:
        df["Voltage"] = pd.to_numeric(df["V_median"], errors="coerce")
        return "Voltage"

    if "V_q25" in df.columns and "V_q75" in df.columns:
        df["Voltage"] = (
            pd.to_numeric(df["V_q25"], errors="coerce")
            + pd.to_numeric(df["V_q75"], errors="coerce")
        ) / 2.0
        return "Voltage"

    raise KeyError(
        "No valid voltage column found: "
        "Voltage_clean / Voltage / V_median / V_q25+V_q75"
    )


def prepare_factor_columns(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    required = ["Temperature", "Rate", "Charge_Rate"]
    missing = [c for c in required if c not in d.columns]

    if missing:
        raise KeyError(f"Required factor columns missing: {missing}")

    d["Temperature"] = pd.to_numeric(d["Temperature"], errors="coerce")
    d["Rate"] = d["Rate"].astype(str).str.strip()
    d["Charge_Rate"] = d["Charge_Rate"].astype(str).str.strip()

    return d


def add_or_repair_condition_id(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    if "Condition_ID" in d.columns:
        d["Condition_ID"] = pd.to_numeric(d["Condition_ID"], errors="coerce")
    else:
        d["Condition_ID"] = np.nan

    missing_mask = d["Condition_ID"].isna()

    if missing_mask.any():
        def infer_id(row):
            try:
                key = (
                    int(row["Temperature"]),
                    str(row["Rate"]).strip(),
                    str(row["Charge_Rate"]).strip(),
                )
                return INV_COND_MAP.get(key, np.nan)
            except Exception:
                return np.nan

        d.loc[missing_mask, "Condition_ID"] = (
            d.loc[missing_mask].apply(infer_id, axis=1)
        )

    d["Condition_ID"] = pd.to_numeric(d["Condition_ID"], errors="coerce")

    return d


def model_ready_dataframe(df: pd.DataFrame, use_bands=USE_SOC_BANDS):
    d = prepare_factor_columns(df)
    vcol = pick_voltage_col(d)

    if "SoC" not in d.columns:
        raise KeyError("Required column missing: SoC")

    d["SoC"] = pd.to_numeric(d["SoC"], errors="coerce")
    d[vcol] = pd.to_numeric(d[vcol], errors="coerce")

    if "SoC_cont" not in d.columns:
        d["SoC_cont"] = d["SoC"] / 100.0
    else:
        d["SoC_cont"] = pd.to_numeric(d["SoC_cont"], errors="coerce")

    band_col = find_col(d, ["SoC_band", "soc_band"])

    if band_col is not None:
        d["SoC_band_std"] = d[band_col].apply(standard_band_value)
    else:
        d["SoC_band_std"] = pd.cut(
            d["SoC"].clip(0, 100),
            bins=[0, 20, 60, 90, 100],
            labels=["0-20", "20-60", "60-90", "90-100"],
            include_lowest=True
        ).astype(str)

    d = d[d["SoC_band_std"].isin(use_bands)].copy()

    required = [vcol, "Temperature", "Rate", "Charge_Rate", "SoC", "SoC_cont"]
    d = d.dropna(subset=required).copy()

    if d.empty:
        raise ValueError(
            "No usable rows left after filtering and dropping missing model fields."
        )

    d["Temperature"] = d["Temperature"].astype(int).astype("category")
    d["Rate"] = d["Rate"].astype("category")
    d["Charge_Rate"] = d["Charge_Rate"].astype("category")

    return d, vcol


def wald_terms(robust_res, model, label: str) -> pd.DataFrame:
    """
    Term-level Wald tests using the fitted formula model's Patsy term slices.
    Compatible with HC3 and cluster-robust covariance results.
    """
    design_info = model.model.data.design_info
    term_slices = getattr(design_info, "term_name_slices", None)

    if term_slices is None:
        raise RuntimeError(
            "DesignInfo.term_name_slices not found. "
            "Please update statsmodels/patsy."
        )

    p = len(model.params)
    rows = []

    for term_name, sl in term_slices.items():
        if term_name.lower() == "intercept":
            continue

        cols = np.arange(sl.start, sl.stop)
        k = len(cols)

        if k == 0:
            continue

        R = np.zeros((k, p))

        for i, cidx in enumerate(cols):
            R[i, cidx] = 1.0

        try:
            wt = robust_res.wald_test(R, use_f=True)
            Fval = float(np.asarray(wt.fvalue).squeeze())
            pval = float(np.asarray(wt.pvalue).squeeze())
            err = ""
        except Exception as e:
            Fval = np.nan
            pval = np.nan
            err = str(e)

        rows.append({
            "Inference": label,
            "Term": term_name,
            "df": int(k),
            "F": Fval,
            "p": pval,
            "error": err,
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# Point-level HC3 ANOVA cell 17
# ============================================================

def run_hc3_anova():
    ensure_exists(TIDY_PATH)

    raw = pd.read_csv(TIDY_PATH)
    d, vcol = model_ready_dataframe(raw, use_bands=USE_SOC_BANDS)

    formula = f"{vcol} ~ C(Temperature)*C(Rate)*C(Charge_Rate) + SoC_cont"

    ols_res = smf.ols(formula, data=d).fit()
    robust = ols_res.get_robustcov_results(cov_type="HC3")

    hc3_terms = wald_terms(robust, ols_res, label="point_level_HC3")

    terms_path = os.path.join(OUT_DIR, "anova_TxIcutxDrate_with_SoC_HC3.csv")
    summary_path = os.path.join(OUT_DIR, "anova_HC3_model_summary.txt")
    note_path = os.path.join(OUT_DIR, "anova_HC3_method_note.txt")

    hc3_terms.to_csv(terms_path, index=False)

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(robust.summary().as_text())

    with open(note_path, "w", encoding="utf-8") as f:
        f.write(
            "HC3-robust inference is computed on point-level voltage records "
            "over the analysed SoC bands.\n"
            "HC3 addresses heteroskedasticity but does not remove within-curve, "
            "serial, or cluster dependence.\n"
            "Therefore, these results should be interpreted as point-level "
            "factorial screening.\n"
            "See anova_bandlevel_cluster_terms.csv for a de-duplicated "
            "band-level cluster-robust sensitivity check.\n"
        )

    print("[OK] Point-level HC3 ANOVA saved:")
    print(" -", terms_path)
    print(f"Rows used: {len(d):,} | Voltage column: {vcol}")

    return {
        "rows_used": int(len(d)),
        "voltage_col": vcol,
        "terms_file": terms_path,
        "summary_file": summary_path,
        "note_file": note_path,
    }

In [ ]:
# ============================================================
# Band-level aggregated HC3 sensitivity ANOVA helpers cell 18
# ============================================================
# Preferred robustness/sensitivity check:
#
#   1. Remove repeated condition-SoC entries created by family-level reuse.
#   2. Collapse dense point-level traces to Condition_ID x SoC-band summaries.
#   3. Run HC3-robust term-level Wald tests on the aggregated band-level table.
#
# This avoids treating thousands of SoC points as independent.
# ============================================================

BAND_INPUT_PATH = os.path.join(OUT_DIR, "anova_bandlevel_input.csv")

def wald_terms_hc3(robust_res, model):
    design_info = model.model.data.design_info
    term_slices = getattr(design_info, "term_name_slices", None)

    if term_slices is None:
        raise RuntimeError("DesignInfo.term_name_slices not found.")

    p = len(model.params)
    rows = []

    for term_name, sl in term_slices.items():
        if term_name.lower() == "intercept":
            continue

        cols = np.arange(sl.start, sl.stop)
        k = len(cols)

        if k == 0:
            continue

        R = np.zeros((k, p))

        for i, cidx in enumerate(cols):
            R[i, cidx] = 1.0

        try:
            wt = robust_res.wald_test(R, use_f=True)
            Fval = float(np.asarray(wt.fvalue).squeeze())
            pval = float(np.asarray(wt.pvalue).squeeze())
            err = ""
        except Exception as e:
            Fval = np.nan
            pval = np.nan
            err = str(e)

        rows.append({
            "Inference": "band_level_aggregated_HC3",
            "Term": term_name,
            "df": int(k),
            "F_HC3": Fval,
            "p_HC3": pval,
            "error": err,
        })

    return pd.DataFrame(rows)


def build_bandlevel_input_from_tidy():
    ensure_exists(TIDY_PATH)

    d = pd.read_csv(TIDY_PATH).copy()
    vcol = pick_voltage_col(d)

    required = {"Temperature", "Rate", "Charge_Rate", "SoC"}
    missing = required - set(d.columns)

    if missing:
        raise KeyError(f"Missing columns in tidy_clean.csv: {missing}")

    d["Temperature"] = pd.to_numeric(d["Temperature"], errors="coerce")
    d["SoC"] = pd.to_numeric(d["SoC"], errors="coerce")
    d[vcol] = pd.to_numeric(d[vcol], errors="coerce")

    d["Rate"] = d["Rate"].astype(str).str.strip()
    d["Charge_Rate"] = d["Charge_Rate"].astype(str).str.strip()

    if "SoC_cont" not in d.columns:
        d["SoC_cont"] = d["SoC"] / 100.0
    else:
        d["SoC_cont"] = pd.to_numeric(d["SoC_cont"], errors="coerce")

    if "SoC_band" in d.columns:
        d["SoC_band_std"] = d["SoC_band"].apply(standard_band_value)
    elif "soc_band" in d.columns:
        d["SoC_band_std"] = d["soc_band"].apply(standard_band_value)
    else:
        d["SoC_band_std"] = pd.cut(
            d["SoC"].clip(0, 100),
            bins=[0, 20, 60, 90, 100],
            labels=["0-20", "20-60", "60-90", "90-100"],
            include_lowest=True
        ).astype(str)

    d = d[d["SoC_band_std"].isin(USE_SOC_BANDS)].copy()
    d = add_or_repair_condition_id(d)

    required2 = [
        "Condition_ID",
        "Temperature",
        "Rate",
        "Charge_Rate",
        "SoC",
        "SoC_cont",
        "SoC_band_std",
        vcol,
    ]

    d = d.dropna(subset=required2).copy()

    if d.empty:
        raise ValueError("No usable rows left for band-level aggregation.")

    d["Condition_ID"] = d["Condition_ID"].astype(int)
    d["Temperature"] = d["Temperature"].astype(int)
    d["Rate"] = d["Rate"].astype(str)
    d["Charge_Rate"] = d["Charge_Rate"].astype(str)
    d["SoC_band_std"] = d["SoC_band_std"].astype(str)

    point_rows_before_dedup = len(d)

    # Remove repeated condition-SoC entries created by family-level reuse.
    d = d.drop_duplicates(
        subset=["Condition_ID", "Temperature", "Rate", "Charge_Rate", "SoC"]
    ).copy()

    point_rows_after_dedup = len(d)

    band = (
        d.groupby(
            ["Condition_ID", "Temperature", "Rate", "Charge_Rate", "SoC_band_std"],
            as_index=False,
            observed=True
        )
        .agg(
            Voltage_band=(vcol, "mean"),
            SoC_cont=("SoC_cont", "mean"),
            n_soc_points=("SoC", "size")
        )
    )

    band = band.dropna(subset=["Voltage_band", "SoC_cont"]).copy()

    if band.empty:
        raise ValueError("Band-level dataframe is empty after aggregation.")

    band.to_csv(BAND_INPUT_PATH, index=False)

    diag = {
        "source": "rebuilt_from_tidy_clean.csv",
        "voltage_col": vcol,
        "point_rows_before_dedup": int(point_rows_before_dedup),
        "point_rows_after_dedup": int(point_rows_after_dedup),
        "bandlevel_rows": int(len(band)),
        "n_condition_clusters": int(band["Condition_ID"].nunique()),
        "band_input_file": BAND_INPUT_PATH,
    }

    return band, diag

In [ ]:
# ============================================================
# Run band-level aggregated HC3 sensitivity ANOVA cell 19
# ============================================================

def run_bandlevel_aggregated_hc3():
    band, diag = build_bandlevel_input_from_tidy()

    required = {
        "Condition_ID",
        "Temperature",
        "Rate",
        "Charge_Rate",
        "SoC_band_std",
        "Voltage_band",
        "SoC_cont",
    }

    missing = required - set(band.columns)

    if missing:
        raise KeyError(f"Missing columns in band-level input: {missing}")

    band = band.dropna(subset=list(required)).copy()

    if len(band) < 40:
        raise ValueError(f"Too few band-level rows: {len(band)}")

    band["Temperature"] = (
        pd.to_numeric(band["Temperature"], errors="coerce")
        .astype(int)
        .astype("category")
    )
    band["Rate"] = band["Rate"].astype(str).astype("category")
    band["Charge_Rate"] = band["Charge_Rate"].astype(str).astype("category")
    band["SoC_cont"] = pd.to_numeric(band["SoC_cont"], errors="coerce")
    band["Voltage_band"] = pd.to_numeric(band["Voltage_band"], errors="coerce")

    band = band.dropna(subset=["Voltage_band", "SoC_cont"]).copy()

    formula = "Voltage_band ~ C(Temperature)*C(Rate)*C(Charge_Rate) + SoC_cont"

    ols_res = smf.ols(formula, data=band).fit()
    hc3_res = ols_res.get_robustcov_results(cov_type="HC3")

    terms = wald_terms_hc3(hc3_res, ols_res)

    terms_path = os.path.join(OUT_DIR, "anova_bandlevel_HC3_terms.csv")
    summary_path = os.path.join(OUT_DIR, "anova_bandlevel_HC3_summary.txt")
    diag_path = os.path.join(OUT_DIR, "anova_bandlevel_HC3_diag.json")
    note_path = os.path.join(OUT_DIR, "anova_bandlevel_HC3_method_note.txt")

    terms.to_csv(terms_path, index=False)

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(hc3_res.summary().as_text())
        f.write("\n\nMethod note:\n")
        f.write(
            "This sensitivity analysis first aggregates the voltage data to "
            "Condition_ID x SoC-band summaries and then applies HC3-robust "
            "term-level Wald tests. It reduces repeated point-level dependence "
            "from dense SoC traces and should be interpreted as a robustness "
            "check for the point-level HC3 factorial screening model.\n"
        )

    full_diag = {
        **diag,
        "model_rows_used": int(len(band)),
        "model_df_resid": float(ols_res.df_resid),
        "model_rank": int(ols_res.df_model + 1),
        "formula": formula,
        "outputs": {
            "terms": terms_path,
            "summary": summary_path,
            "diag": diag_path,
            "method_note": note_path,
        },
    }

    with open(diag_path, "w", encoding="utf-8") as f:
        json.dump(full_diag, f, indent=2, ensure_ascii=False)

    with open(note_path, "w", encoding="utf-8") as f:
        f.write(
            "Use anova_bandlevel_HC3_terms.csv as the preferred dependence-reduced "
            "sensitivity check. Do not report the previous cluster-by-Condition_ID "
            "F statistics as primary inferential values because the cluster variable "
            "is nearly identical to the treatment tuple.\n"
        )

    print("[OK] Band-level aggregated HC3 sensitivity ANOVA completed.")
    print(f"     Band-level rows used: {len(band):,}")
    print(f"     Residual df: {ols_res.df_resid}")
    print(f"     Terms file: {terms_path}")
    print(f"     Diagnostic file: {diag_path}")

    return full_diag, terms

In [ ]:
# ============================================================
# Run HC3, band-level robustness, and bucket validation cell 20
# ============================================================

run_info = {
    "inputs": {
        "tidy_clean.csv": os.path.exists(TIDY_PATH),
        "delta_cutoff_T25_DR1C.csv": os.path.exists(DELTA_CUTOFF),
        "delta_ratehist_T25_R_C40.csv": os.path.exists(DELTA_RATEHIST),
        "delta_temp_R_C40_DR1C.csv": os.path.exists(DELTA_TEMP),
    },
    "outputs": {},
    "warnings": [],
}

# 1) Point-level HC3 ANOVA
try:
    run_info["outputs"]["point_level_HC3"] = run_hc3_anova()
except Exception as e:
    msg = f"HC3 ANOVA failed: {e}"
    print("[WARN]", msg)
    run_info["warnings"].append(msg)

# 2) Preferred band-level aggregated HC3 sensitivity check
try:
    diag_hc3, terms_hc3 = run_bandlevel_aggregated_hc3()
    run_info["outputs"]["bandlevel_aggregated_HC3"] = diag_hc3
except Exception as e:
    msg = f"Band-level aggregated HC3 sensitivity check failed: {e}"
    print("[WARN]", msg)
    run_info["warnings"].append(msg)

# 3) Practical bucket summaries
try:
    run_info["outputs"]["bucket_cutoff"] = validate_and_summarize_buckets(
        delta_csv_path=DELTA_CUTOFF,
        checked_delta_path=os.path.join(OUT_DIR, "delta_cutoff_T25_DR1C_bucket_checked.csv"),
        out_csv_path=os.path.join(OUT_DIR, "bucket_summary_cutoff.csv"),
        legacy_out_csv_path=None,
    )
except Exception as e:
    msg = f"Cut-off bucket summary failed: {e}"
    print("[WARN]", msg)
    run_info["warnings"].append(msg)

try:
    run_info["outputs"]["bucket_ratehist"] = validate_and_summarize_buckets(
        delta_csv_path=DELTA_RATEHIST,
        checked_delta_path=os.path.join(OUT_DIR, "delta_ratehist_T25_R_C40_bucket_checked.csv"),
        out_csv_path=os.path.join(OUT_DIR, "bucket_summary_ratehist.csv"),
        legacy_out_csv_path=None,
    )
except Exception as e:
    msg = f"Rate/history bucket summary failed: {e}"
    print("[WARN]", msg)
    run_info["warnings"].append(msg)

try:
    run_info["outputs"]["bucket_temp"] = validate_and_summarize_buckets(
        delta_csv_path=DELTA_TEMP,
        checked_delta_path=os.path.join(OUT_DIR, "delta_temp_R_C40_DR1C_bucket_checked.csv"),
        out_csv_path=os.path.join(OUT_DIR, "bucket_summary_temp.csv"),
        legacy_out_csv_path=None,
    )
except Exception as e:
    msg = f"Thermal bucket summary failed: {e}"
    print("[WARN]", msg)
    run_info["warnings"].append(msg)

method_note = (
    "Point-level HC3 ANOVA is retained as a heteroskedasticity-robust "
    "factorial screening analysis. Because multiple SoC points may originate "
    "from the same condition-level voltage profiles, HC3 alone does not remove "
    "within-curve, serial, or cluster dependence. Therefore, this notebook also "
    "reports a de-duplicated band-level cluster-robust sensitivity check by "
    "Condition_ID. The manuscript should describe the point-level ANOVA as "
    "descriptive factorial screening and use the band-level cluster-robust output "
    "as a robustness/sensitivity check."
)

method_note_path = os.path.join(OUT_DIR, "anova_dependence_method_note.txt")

with open(method_note_path, "w", encoding="utf-8") as f:
    f.write(method_note + "\n")

run_info["method_note"] = method_note

run_json_path = os.path.join(OUT_DIR, "sec5_hc3_buckets_cluster_run.json")

with open(run_json_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)

print("\n[RUN SUMMARY]")
print(json.dumps(run_info, indent=2, ensure_ascii=False))
print("\n[OK] Run metadata saved:")
print(run_json_path)

In [ ]:
# =========================================================
# Section 5 forest plots — corrected terminology + bucket validation  cell 22
#
# Produces:
#   forest_cutoff_T25_CR1C.png
#   forest_ratehist_T25_R_C40.png
#   forest_temp_R_C40_CR1C.png
#
# Terminology:
#   Rate = charge cut-off current: C/5 vs C/40
#   Charge_Rate = discharge-rate/rate-history level: 0.7C, 1C, 2C
# =========================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------- Path discovery --------------------
SEARCH_BASES = [
    "/content/sec5_outputs",
    "/content",
    "/mnt/data/sec5_outputs",
    "/mnt/data",
    "sec5_outputs",
    ".",
]

def find_first(existing_rel_names):
    if isinstance(existing_rel_names, str):
        existing_rel_names = [existing_rel_names]

    for rel in existing_rel_names:
        for base in SEARCH_BASES:
            p = os.path.join(base, rel)
            if os.path.exists(p):
                return os.path.abspath(p)

    raise FileNotFoundError(
        f"Could not find any of {existing_rel_names} in {SEARCH_BASES}"
    )

# Prefer bucket-checked files. Fall back to original delta files if needed.
cutoff_csv = find_first([
    "sec5_outputs/delta_cutoff_T25_CR1C_bucket_checked.csv",
    "delta_cutoff_T25_CR1C_bucket_checked.csv",
    "sec5_outputs/delta_cutoff_bucket_checked.csv",
    "delta_cutoff_bucket_checked.csv",
    "sec5_outputs/delta_rate_T25_CR1C.csv",
    "delta_rate_T25_CR1C.csv",
])

ratehist_csv = find_first([
    "sec5_outputs/delta_ratehist_T25_R_C40_bucket_checked.csv",
    "delta_ratehist_T25_R_C40_bucket_checked.csv",
    "sec5_outputs/delta_ratehist_bucket_checked.csv",
    "delta_ratehist_bucket_checked.csv",
    "sec5_outputs/delta_charge_T25_R_C40.csv",
    "delta_charge_T25_R_C40.csv",
])

temp_csv = find_first([
    "sec5_outputs/delta_temp_R_C40_CR1C_bucket_checked.csv",
    "delta_temp_R_C40_CR1C_bucket_checked.csv",
    "sec5_outputs/delta_temp_bucket_checked.csv",
    "delta_temp_bucket_checked.csv",
    "sec5_outputs/delta_temp_R_C40_CR1C.csv",
    "delta_temp_R_C40_CR1C.csv",
])

# Output directory
OUT_DIR_CANDIDATES = [
    "/content/sec5_outputs",
    "/mnt/data/sec5_outputs",
    "sec5_outputs",
]

OUT_DIR = None
for d in OUT_DIR_CANDIDATES:
    try:
        os.makedirs(d, exist_ok=True)
        if os.path.isdir(d):
            OUT_DIR = d
            break
    except Exception:
        pass

if OUT_DIR is None:
    OUT_DIR = "sec5_outputs"
    os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILES = {
    "cutoff":   os.path.join(OUT_DIR, "forest_cutoff_T25_CR1C.png"),
    "ratehist": os.path.join(OUT_DIR, "forest_ratehist_T25_R_C40.png"),
    "temp":     os.path.join(OUT_DIR, "forest_temp_R_C40_CR1C.png"),
}

print("Found inputs:")
print(" - charge cut-off current:", cutoff_csv)
print(" - rate/history          :", ratehist_csv)
print(" - thermal               :", temp_csv)
print("Output dir:", os.path.abspath(OUT_DIR))

# -------------------- Plot helpers --------------------
SOC_ORDER = ["0-20", "20-60", "60-90"]
BUCKET_MARKER = {
    "negligible": "o",
    "small": "s",
    "moderate": "D",
    "large": "^",
}

THRESH_SMALL  = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE  = 0.15

def practical_bucket(delta_abs):
    if pd.isna(delta_abs):
        return "NA"
    v = float(abs(delta_abs))
    if v >= THRESH_LARGE:
        return "large"
    if v >= THRESH_MEDIUM:
        return "moderate"
    if v >= THRESH_SMALL:
        return "small"
    return "negligible"

def standard_band_value(x):
    s = str(x).strip()
    for ch in ["[", "]", "(", ")"]:
        s = s.replace(ch, "")
    s = s.replace(", ", "-").replace(",", "-")
    s = s.replace("0.0-20.0", "0-20")
    s = s.replace("20.0-60.0", "20-60")
    s = s.replace("60.0-90.0", "60-90")
    s = s.replace("90.0-100.0", "90-100")
    return s

def load_delta_csv(path):
    df = pd.read_csv(path)

    # Accept either soc_band or SoC_band.
    if "soc_band" in df.columns:
        band_col = "soc_band"
    elif "SoC_band" in df.columns:
        band_col = "SoC_band"
    else:
        raise KeyError(f"Missing SoC band column in {path}: expected soc_band or SoC_band")

    required = {"mean_diff", "ci_low", "ci_high"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns in {path}: {missing}")

    df["soc_band"] = df[band_col].apply(standard_band_value)
    df = df[df["soc_band"].isin(SOC_ORDER)].copy()
    df["soc_band"] = pd.Categorical(df["soc_band"], categories=SOC_ORDER, ordered=True)
    df = df.sort_values("soc_band")

    for c in ["mean_diff", "ci_low", "ci_high"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Recompute bucket from mean_diff to avoid manuscript/table inconsistencies.
    df["practical_bucket_plot"] = df["mean_diff"].abs().apply(practical_bucket)

    # Symmetric/asymmetric CI error bars.
    df["err_lo"] = np.maximum(df["mean_diff"] - df["ci_low"], 0)
    df["err_hi"] = np.maximum(df["ci_high"] - df["mean_diff"], 0)

    return df

def forest_plot(df, title, outfile):
    y = np.arange(len(df))[::-1]
    x = df["mean_diff"].to_numpy(float)

    err_lo = df["err_lo"].to_numpy(float)
    err_hi = df["err_hi"].to_numpy(float)
    xerr = np.vstack([err_lo, err_hi])

    labels = df["soc_band"].astype(str).tolist()
    buckets = df["practical_bucket_plot"].astype(str).str.lower().str.strip().tolist()
    markers = [BUCKET_MARKER.get(b, "o") for b in buckets]

    plt.figure(figsize=(5.4, 3.6))
    plt.axvline(0, linestyle="--", linewidth=1, alpha=0.6)

    for i, (xi, yi, mi) in enumerate(zip(x, y, markers)):
        plt.errorbar(
            xi,
            yi,
            xerr=[[xerr[0, i]], [xerr[1, i]]],
            fmt="none",
            linewidth=1,
            capsize=3,
        )
        plt.plot([xi], [yi], marker=mi, markersize=6)

    plt.yticks(y, labels)
    plt.xlabel(r"$\Delta V$ (V)")
    plt.title(title, fontsize=10.5)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"[Saved] {os.path.abspath(outfile)}")

# -------------------- Generate figures --------------------
df_cutoff = load_delta_csv(cutoff_csv)
forest_plot(
    df_cutoff,
    "Charge cut-off current: C/40 − C/5 @ 25°C, 1C",
    OUT_FILES["cutoff"],
)

df_ratehist = load_delta_csv(ratehist_csv)
forest_plot(
    df_ratehist,
    "Mild discharge-rate/rate-history: 0.7C − 1C @ 25°C, C/40",
    OUT_FILES["ratehist"],
)

df_temp = load_delta_csv(temp_csv)
forest_plot(
    df_temp,
    "Thermal contrast: 60°C − 25°C @ C/40, 1C",
    OUT_FILES["temp"],
)

print("\nFiles ready:")
for k, p in OUT_FILES.items():
    print(f" - {k}: {os.path.abspath(p)}")

print("\nBucket check used for plotting:")
for name, df in [
    ("cutoff", df_cutoff),
    ("ratehist", df_ratehist),
    ("temp", df_temp),
]:
    print(f"\n{name}:")
    print(df[["soc_band", "mean_diff", "ci_low", "ci_high", "practical_bucket_plot"]].to_string(index=False))

In [ ]:
## Expected outputs

Main output directory:

`outputs/04_section5_statistical_analysis/`

Core outputs generated by this notebook:

- `tidy_clean.csv`
- `mask_report_by_group.csv`
- `delta_cutoff_T25_DR1C.csv`
- `delta_ratehist_T25_R_C40.csv`
- `delta_temp_R_C40_DR1C.csv`
- `heatmap_delta_cutoff_grid.csv`
- `heatmap_delta_ratehist_grid.csv`
- `heatmap_delta_temp_grid.csv`
- `anova_TxIcutxDrate_with_SoC.csv`
- `anova_model_summary.txt`
- `run_summary.json`

Robustness and bucket-extension outputs:

- `anova_TxIcutxDrate_with_SoC_HC3.csv`
- `anova_HC3_model_summary.txt`
- `anova_HC3_method_note.txt`
- `anova_bandlevel_input.csv`
- `anova_bandlevel_HC3_terms.csv`
- `anova_bandlevel_HC3_summary.txt`
- `anova_bandlevel_HC3_diag.json`
- `anova_bandlevel_HC3_method_note.txt`
- `delta_cutoff_T25_DR1C_bucket_checked.csv`
- `delta_ratehist_T25_R_C40_bucket_checked.csv`
- `delta_temp_R_C40_DR1C_bucket_checked.csv`
- `bucket_summary_cutoff.csv`
- `bucket_summary_ratehist.csv`
- `bucket_summary_temp.csv`
- `anova_dependence_method_note.txt`
- `sec5_hc3_buckets_cluster_run.json`

Forest-plot outputs:

- `forest_cutoff_T25_DR1C.png`
- `forest_ratehist_T25_R_C40.png`
- `forest_temp_R_C40_DR1C.png`

Interpretation note:

The point-level ANOVA and point-level HC3 outputs are retained as dense factorial-screening summaries. The band-level aggregated HC3 output is the preferred dependence-reduced robustness check. Forest plots visualize the canonical voltage-shift contrasts and recompute practical-bucket labels from numerical `mean_diff` values before plotting.

In [ ]:
# ============================================================
# Section 5 - Practical significance summary, RateHist 3-level
# summary, and robustness mini-table
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   04_section5_statistical_analysis.ipynb
#
# Reads:
#   outputs/04_section5_statistical_analysis/tidy_clean.csv
#   outputs/04_section5_statistical_analysis/heatmap_delta_cutoff_grid.csv
#   outputs/04_section5_statistical_analysis/heatmap_delta_ratehist_grid.csv
#   outputs/04_section5_statistical_analysis/heatmap_delta_temp_grid.csv
#
# Outputs:
#   outputs/04_section5_statistical_analysis/practical_significance_summary.csv
#   outputs/04_section5_statistical_analysis/ratehist_three_level_summary.csv
#   outputs/04_section5_statistical_analysis/charge_three_level_summary_legacy_do_not_use.csv
#   outputs/04_section5_statistical_analysis/robustness_mini.csv
#   outputs/04_section5_statistical_analysis/robustness_mini_method_note.json
#
# Terminology:
#   Rate        = charge cut-off current: C/5, C/40
#   Charge_Rate = discharge-rate/rate-history level: 0.7C, 1C, 2C
# ============================================================

import os
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OUT_DIR = "outputs/04_section5_statistical_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

TIDY_PATH = os.path.join(OUT_DIR, "tidy_clean.csv")

CUTOFF_GRID_CANDIDATES = [
    os.path.join(OUT_DIR, "heatmap_delta_cutoff_grid.csv"),
    os.path.join(OUT_DIR, "heatmap_delta_rate_grid.csv"),  # legacy
]

RATEHIST_GRID_CANDIDATES = [
    os.path.join(OUT_DIR, "heatmap_delta_ratehist_grid.csv"),
    os.path.join(OUT_DIR, "heatmap_delta_charge_grid.csv"),  # legacy
]

TEMP_GRID_CANDIDATES = [
    os.path.join(OUT_DIR, "heatmap_delta_temp_grid.csv"),
]


def find_existing_path(candidates, required=False):
    for path in candidates:
        if os.path.exists(path):
            return path

    if required:
        raise FileNotFoundError(
            "None of the required files were found:\n"
            + "\n".join(candidates)
        )

    return None


if not os.path.exists(TIDY_PATH):
    raise FileNotFoundError(
        "tidy_clean.csv was not found. Run the main Section 5 pipeline first:\n"
        f"{TIDY_PATH}"
    )

cutoff_grid_path = find_existing_path(CUTOFF_GRID_CANDIDATES, required=False)
ratehist_grid_path = find_existing_path(RATEHIST_GRID_CANDIDATES, required=False)
temp_grid_path = find_existing_path(TEMP_GRID_CANDIDATES, required=False)

print("Section 5 summary inputs:")
print(" - tidy_clean:", TIDY_PATH)
print(" - cut-off grid:", cutoff_grid_path)
print(" - RateHist grid:", ratehist_grid_path)
print(" - thermal grid:", temp_grid_path)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

SOC_BINS_BASE = [0, 20, 60, 90, 100]
SOC_LABELS_BASE = ["0-20", "20-60", "60-90", "90-100"]
USE_SOC_BANDS = ["0-20", "20-60", "60-90"]

THRESH_SMALL = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE = 0.15


def practical_bucket(delta_abs):
    if pd.isna(delta_abs) or not np.isfinite(delta_abs):
        return "NA"

    value = float(abs(delta_abs))

    if value >= THRESH_LARGE:
        return "large"
    if value >= THRESH_MEDIUM:
        return "moderate"
    if value >= THRESH_SMALL:
        return "small"

    return "negligible"


def standard_band_value(x):
    s = str(x).strip()

    for ch in ["[", "]", "(", ")"]:
        s = s.replace(ch, "")

    s = s.replace(", ", "-").replace(",", "-")
    s = s.replace("0.0-20.0", "0-20")
    s = s.replace("20.0-60.0", "20-60")
    s = s.replace("60.0-90.0", "60-90")
    s = s.replace("90.0-100.0", "90-100")

    return s


def load_csv_optional(path):
    if path is None:
        return None

    return pd.read_csv(path)


def normalize_columns(df):
    df = df.copy()
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    def pick(candidates=None, contains=None):
        if candidates:
            for cand in candidates:
                key = cand.lower()
                if key in lower:
                    return lower[key]

        if contains:
            for c in cols:
                if contains.lower() in c.lower():
                    return c

        return None

    colmap = {}

    soc = pick(candidates=["SoC", "soc", "state_of_charge"], contains="soc")
    if soc:
        colmap[soc] = "SoC"

    temp = pick(
        candidates=["Temperature", "Temp", "ambient_temperature", "T"],
        contains="temp"
    )
    if temp:
        colmap[temp] = "Temperature"

    if "Rate" not in df.columns:
        rate = pick(candidates=["Rate", "Discharge_Rate", "Discharge"])
        if rate:
            colmap[rate] = "Rate"

    if "Charge_Rate" not in df.columns:
        charge_rate = pick(
            candidates=["Charge_Rate", "ChargeRate", "Charge-Rate", "CR"],
            contains="charge"
        )
        if charge_rate:
            colmap[charge_rate] = "Charge_Rate"

    if "Voltage_clean" in df.columns:
        pass
    elif "V_median" in df.columns:
        colmap["V_median"] = "Voltage"
    else:
        voltage = pick(
            candidates=["Voltage", "VoltageV", "V", "Cell_Voltage", "Voltage (V)"],
            contains="volt"
        )
        if voltage:
            colmap[voltage] = "Voltage"

    return df.rename(columns=colmap)


def pick_voltage_col(df):
    if "Voltage_clean" in df.columns:
        return "Voltage_clean"

    if "Voltage" in df.columns:
        return "Voltage"

    if "V_median" in df.columns:
        df["Voltage"] = pd.to_numeric(df["V_median"], errors="coerce")
        return "Voltage"

    if "V_q25" in df.columns and "V_q75" in df.columns:
        df["Voltage"] = (
            pd.to_numeric(df["V_q25"], errors="coerce")
            + pd.to_numeric(df["V_q75"], errors="coerce")
        ) / 2.0
        return "Voltage"

    raise KeyError("No valid voltage column found.")


# ------------------------------------------------------------
# Load tidy data
# ------------------------------------------------------------

tidy = pd.read_csv(TIDY_PATH)
tidy = normalize_columns(tidy)

VCOL = pick_voltage_col(tidy)

required = ["SoC", "Temperature", "Rate", "Charge_Rate", VCOL]
missing = [c for c in required if c not in tidy.columns]

if missing:
    raise KeyError(f"Missing required columns in tidy_clean.csv: {missing}")

tidy["SoC"] = pd.to_numeric(tidy["SoC"], errors="coerce")
tidy["Temperature"] = pd.to_numeric(tidy["Temperature"], errors="coerce")
tidy[VCOL] = pd.to_numeric(tidy[VCOL], errors="coerce")

tidy["Rate"] = tidy["Rate"].astype(str).str.strip()
tidy["Charge_Rate"] = tidy["Charge_Rate"].astype(str).str.strip()

tidy = tidy.dropna(
    subset=["SoC", "Temperature", VCOL, "Rate", "Charge_Rate"]
).copy()

tidy["Temperature"] = tidy["Temperature"].astype(int)

if "SoC_band" in tidy.columns:
    tidy["SoC_band"] = tidy["SoC_band"].apply(standard_band_value)
elif "soc_band" in tidy.columns:
    tidy["SoC_band"] = tidy["soc_band"].apply(standard_band_value)
else:
    tidy["SoC_band"] = pd.cut(
        tidy["SoC"].clip(0, 100),
        bins=SOC_BINS_BASE,
        labels=SOC_LABELS_BASE,
        include_lowest=True,
    ).astype(str)

cutoff_grid = load_csv_optional(cutoff_grid_path)
ratehist_grid = load_csv_optional(ratehist_grid_path)
temp_grid = load_csv_optional(temp_grid_path)

# ============================================================
# 1) Practical significance summary
# ============================================================

bucket_counts = {
    band: {"negligible": 0, "small": 0, "moderate": 0, "large": 0}
    for band in USE_SOC_BANDS
}

total_cells = {band: 0 for band in USE_SOC_BANDS}


def add_grid_to_bucket_summary(df, source_name):
    if df is None:
        print(f"[SKIP] {source_name}: grid file not found.")
        return

    if "SoC_band" in df.columns:
        band_col = "SoC_band"
    elif "soc_band" in df.columns:
        band_col = "soc_band"
    else:
        print(f"[SKIP] {source_name}: no SoC-band column.")
        return

    if "mean_diff" not in df.columns:
        print(f"[SKIP] {source_name}: no mean_diff column.")
        return

    d = df.copy()
    d["SoC_band_std"] = d[band_col].apply(standard_band_value)
    d["mean_diff"] = pd.to_numeric(d["mean_diff"], errors="coerce")

    for _, row in d.iterrows():
        band = str(row["SoC_band_std"])

        if band not in USE_SOC_BANDS:
            continue

        mean_diff = row["mean_diff"]

        if pd.isna(mean_diff):
            continue

        bucket = practical_bucket(abs(float(mean_diff)))

        if bucket != "NA":
            bucket_counts[band][bucket] += 1
            total_cells[band] += 1


add_grid_to_bucket_summary(cutoff_grid, "charge cut-off current grid")
add_grid_to_bucket_summary(ratehist_grid, "discharge-rate/rate-history grid")
add_grid_to_bucket_summary(temp_grid, "thermal grid")

summary_rows = []

for band in USE_SOC_BANDS:
    total = total_cells[band]

    row = {
        "SoC_band": band,
        "Negligible_n": bucket_counts[band]["negligible"],
        "Small_n": bucket_counts[band]["small"],
        "Moderate_n": bucket_counts[band]["moderate"],
        "Large_n": bucket_counts[band]["large"],
        "Total_cells": total,
    }

    for label in ["Negligible", "Small", "Moderate", "Large"]:
        n = row[f"{label}_n"]
        row[f"{label}_pct"] = round(100.0 * n / total, 1) if total > 0 else np.nan

    summary_rows.append(row)

practical_summary = pd.DataFrame(summary_rows)

practical_summary_path = os.path.join(
    OUT_DIR,
    "practical_significance_summary.csv"
)

practical_summary.to_csv(practical_summary_path, index=False)

print("\n=== Practical significance summary ===")
display(practical_summary)
print("Saved:", practical_summary_path)

# ============================================================
# 2) RateHist 3-level summary
# ============================================================
# Three-level discharge-rate/rate-history contrast at:
#   Temperature = 25C
#   charge cut-off current = C/40
#
# Computed contrasts:
#   0.7C - 1C
#   2C   - 1C
#   0.7C - 2C

def delta_by_band(df, factor, baseline, contrast, fixed, vcol, use_bands=USE_SOC_BANDS):
    sub = df.copy()

    for key, value in fixed.items():
        sub = sub[sub[key] == value]

    records = []

    for band in use_bands:
        sb = sub[sub["SoC_band"] == band]

        baseline_values = sb[sb[factor] == baseline][vcol].dropna().to_numpy(dtype=float)
        contrast_values = sb[sb[factor] == contrast][vcol].dropna().to_numpy(dtype=float)

        if baseline_values.size > 0 and contrast_values.size > 0:
            mean_diff = float(np.mean(contrast_values) - np.mean(baseline_values))
            bucket = practical_bucket(abs(mean_diff))
        else:
            mean_diff = np.nan
            bucket = "NA"

        records.append({
            "SoC_band": band,
            "baseline": baseline,
            "contrast": contrast,
            "n_baseline": int(baseline_values.size),
            "n_contrast": int(contrast_values.size),
            "mean_diff": mean_diff,
            "bucket": bucket,
        })

    return pd.DataFrame(records)


fixed_ratehist = {
    "Temperature": 25,
    "Rate": "C/40",
}

delta_07_minus_1 = delta_by_band(
    tidy,
    factor="Charge_Rate",
    baseline="1C",
    contrast="0.7C",
    fixed=fixed_ratehist,
    vcol=VCOL,
)

delta_2_minus_1 = delta_by_band(
    tidy,
    factor="Charge_Rate",
    baseline="1C",
    contrast="2C",
    fixed=fixed_ratehist,
    vcol=VCOL,
)

delta_07_minus_2 = delta_by_band(
    tidy,
    factor="Charge_Rate",
    baseline="2C",
    contrast="0.7C",
    fixed=fixed_ratehist,
    vcol=VCOL,
)


def dataframe_to_band_map(df):
    return {row["SoC_band"]: row for _, row in df.iterrows()}


map_07_1 = dataframe_to_band_map(delta_07_minus_1)
map_2_1 = dataframe_to_band_map(delta_2_minus_1)
map_07_2 = dataframe_to_band_map(delta_07_minus_2)

ratehist_rows = []

for band in USE_SOC_BANDS:
    r1 = map_07_1.get(band, {})
    r2 = map_2_1.get(band, {})
    r3 = map_07_2.get(band, {})

    ratehist_rows.append({
        "SoC_band": band,

        "DeltaV_0.7C_minus_1C": r1.get("mean_diff", np.nan),
        "Bucket_0.7C_minus_1C": r1.get("bucket", "NA"),
        "n_1C_for_0.7C_minus_1C": r1.get("n_baseline", 0),
        "n_0.7C_for_0.7C_minus_1C": r1.get("n_contrast", 0),

        "DeltaV_2C_minus_1C": r2.get("mean_diff", np.nan),
        "Bucket_2C_minus_1C": r2.get("bucket", "NA"),
        "n_1C_for_2C_minus_1C": r2.get("n_baseline", 0),
        "n_2C_for_2C_minus_1C": r2.get("n_contrast", 0),

        "DeltaV_0.7C_minus_2C": r3.get("mean_diff", np.nan),
        "Bucket_0.7C_minus_2C": r3.get("bucket", "NA"),
        "n_2C_for_0.7C_minus_2C": r3.get("n_baseline", 0),
        "n_0.7C_for_0.7C_minus_2C": r3.get("n_contrast", 0),
    })

ratehist_three = pd.DataFrame(ratehist_rows)

ratehist_three_path = os.path.join(
    OUT_DIR,
    "ratehist_three_level_summary.csv"
)

legacy_ratehist_three_path = os.path.join(
    OUT_DIR,
    "charge_three_level_summary_legacy_do_not_use.csv"
)

ratehist_three.to_csv(ratehist_three_path, index=False)
ratehist_three.to_csv(legacy_ratehist_three_path, index=False)

print("\n=== RateHist three-level summary @ T=25C, charge cut-off current=C/40 ===")
display(ratehist_three)
print("Saved:", ratehist_three_path)

# ============================================================
# 3) Robustness mini-table
# ============================================================
# Canonical robustness target:
#   charge cut-off current contrast C/40 - C/5
#   at T=25C and discharge-rate/rate-history level=1C.

def mask_top_end(df, top_frac=0.05):
    threshold = 100.0 * (1.0 - float(top_frac))
    return df["SoC"] >= threshold


def recompute_clean_top_only(df, top_frac, vcol):
    d = df.copy()
    d["mask_top_var"] = mask_top_end(d, top_frac)

    values = d[vcol].astype(float).to_numpy()
    clean_values = values.copy()
    clean_values[d["mask_top_var"].to_numpy()] = np.nan

    d["Voltage_clean_topOnly"] = clean_values

    return d


def set_soc_bands(df, edges, labels):
    d = df.copy()

    d["SoC_band"] = pd.cut(
        d["SoC"].clip(0, 100),
        bins=edges,
        labels=labels,
        include_lowest=True,
    ).astype(str)

    return d


def canonical_cutoff_decisions(df, band_labels, vcol="Voltage_clean_topOnly"):
    sub = df[
        (df["Temperature"] == 25)
        & (df["Charge_Rate"] == "1C")
    ].copy()

    records = []

    for band in band_labels:
        sb = sub[sub["SoC_band"] == band]

        c5 = sb[sb["Rate"] == "C/5"][vcol].dropna().to_numpy(dtype=float)
        c40 = sb[sb["Rate"] == "C/40"][vcol].dropna().to_numpy(dtype=float)

        if c5.size > 0 and c40.size > 0:
            mean_diff = float(np.mean(c40) - np.mean(c5))
            sign = "pos" if mean_diff > 0 else ("neg" if mean_diff < 0 else "zero")
            bucket = practical_bucket(abs(mean_diff))
        else:
            mean_diff = np.nan
            sign = "NA"
            bucket = "NA"

        records.append({
            "band": band,
            "mean_diff": mean_diff,
            "sign": sign,
            "bucket": bucket,
            "n_C5": int(c5.size),
            "n_C40": int(c40.size),
        })

    return records


def compare_decisions_by_position(base_decisions, new_decisions):
    n = min(len(base_decisions), len(new_decisions))

    if n == 0:
        return np.nan, np.nan, 0

    same_sign = []
    same_bucket = []

    for i in range(n):
        base_row = base_decisions[i]
        new_row = new_decisions[i]

        if base_row["sign"] != "NA" and new_row["sign"] != "NA":
            same_sign.append(
                1.0 if base_row["sign"] == new_row["sign"] else 0.0
            )

        if base_row["bucket"] != "NA" and new_row["bucket"] != "NA":
            same_bucket.append(
                1.0 if base_row["bucket"] == new_row["bucket"] else 0.0
            )

    sign_pct = round(100.0 * np.mean(same_sign), 1) if same_sign else np.nan
    bucket_pct = round(100.0 * np.mean(same_bucket), 1) if same_bucket else np.nan

    return sign_pct, bucket_pct, n


base = tidy.copy()

base_5 = recompute_clean_top_only(base, top_frac=0.05, vcol=VCOL)
base_5 = set_soc_bands(base_5, SOC_BINS_BASE, SOC_LABELS_BASE)

base_decisions = canonical_cutoff_decisions(base_5, USE_SOC_BANDS)

robust_rows = []

# Band-boundary sensitivity:
# move the 20% boundary to 15% and 25%.
for boundary20 in [15, 25]:
    edges = [0, boundary20, 60, 90, 100]
    labels = [
        f"0-{boundary20}",
        f"{boundary20}-60",
        "60-90",
        "90-100",
    ]

    d = recompute_clean_top_only(base, top_frac=0.05, vcol=VCOL)
    d = set_soc_bands(d, edges, labels)

    new_band_labels = labels[:3]
    decisions = canonical_cutoff_decisions(d, new_band_labels)

    sign_pct, bucket_pct, compared_bands = compare_decisions_by_position(
        base_decisions,
        decisions
    )

    robust_rows.append({
        "Setting": f"20pct boundary -> {boundary20}pct",
        "Same_sign_pct": sign_pct,
        "Same_bucket_pct": bucket_pct,
        "Compared_bands": int(compared_bands),
    })

# Mask sweep: top 3%, 5%, 7%.
for top_frac in [0.03, 0.05, 0.07]:
    d = recompute_clean_top_only(base, top_frac=top_frac, vcol=VCOL)
    d = set_soc_bands(d, SOC_BINS_BASE, SOC_LABELS_BASE)

    decisions = canonical_cutoff_decisions(d, USE_SOC_BANDS)

    sign_pct, bucket_pct, compared_bands = compare_decisions_by_position(
        base_decisions,
        decisions
    )

    robust_rows.append({
        "Setting": f"Mask top {int(round(top_frac * 100))}%",
        "Same_sign_pct": sign_pct,
        "Same_bucket_pct": bucket_pct,
        "Compared_bands": int(compared_bands),
    })

robustness_mini = pd.DataFrame(robust_rows)

robustness_mini_path = os.path.join(
    OUT_DIR,
    "robustness_mini.csv"
)

robustness_mini.to_csv(robustness_mini_path, index=False)

method_note = {
    "robustness_target": (
        "canonical charge cut-off current contrast C/40-C/5 at T=25C "
        "and discharge-rate/rate-history level=1C"
    ),
    "baseline_bands": USE_SOC_BANDS,
    "band_boundary_checks": [
        "20pct boundary -> 15pct",
        "20pct boundary -> 25pct",
    ],
    "mask_checks": [
        "top 3pct",
        "top 5pct",
        "top 7pct",
    ],
    "comparison_rule": (
        "decisions are compared by band position: low, mid, high; "
        "not by literal band label"
    ),
}

method_note_path = os.path.join(
    OUT_DIR,
    "robustness_mini_method_note.json"
)

with open(method_note_path, "w", encoding="utf-8") as f:
    json.dump(method_note, f, indent=2, ensure_ascii=False)

print("\n=== Robustness mini-table: canonical charge cut-off current contrast ===")
display(robustness_mini)

print("\nSaved files:")
print(practical_summary_path)
print(ratehist_three_path)
print(legacy_ratehist_three_path)
print(robustness_mini_path)
print(method_note_path)